In [34]:
import pandas as pd

In [35]:
df = pd.read_json("C:\\Users\\hanib\\Desktop\\nlp_final_final_final_final_project\\Abstract-Evaluator\\data\\openreview\\iclr2024_openreview_7000.json")

print(df.shape)
print(df.columns.tolist())
df.head()

(7000, 10)
['paper_id', 'number', 'title', 'abstract', 'keywords', 'pdf', 'n_reviews', 'reviews', 'decision', 'decision_comment']


,paper_id,number,title,abstract,keywords,pdf,n_reviews,reviews,decision,decision_comment
0,cXs5md5wAq,9504,Modelling Microbial Communities with Graph Neu...,Understanding the interactions and interplay o...,"[graph neural networks, microbial communities,...",/pdf/a4578db3b369ee02db6e42b64d333e578e1b692e.pdf,4,"[{'rating': '3: reject, not good enough', 'con...",Reject,
1,rhgIgTSSxW,9502,TabR: Tabular Deep Learning Meets Nearest Neig...,Deep learning (DL) models for tabular data pro...,"[tabular, tabular data, architecture, deep lea...",/pdf/178e173a880d7872c0a79d88e005426c20501329.pdf,4,"[{'rating': '8: accept, good paper', 'confiden...",Accept (poster),
2,kKRbAY4CXv,9498,Neural Evolutionary Kernel Method: A Knowledge...,Numerical solution of partial differential equ...,"[Numerical PDE, structure preserving neural ne...",/pdf/c330ae354c1b65e4afaa1f53a1ca188d24fcf27f.pdf,4,[{'rating': '6: marginally above the acceptanc...,Reject,
3,ApjY32f3Xr,9493,PINNacle: A Comprehensive Benchmark of Physics...,While significant progress has been made on Ph...,"[PINN, machine learning, physics-informed mach...",/pdf/49840b2f19f2bbeff0c1539d86c876d01140da89.pdf,4,[{'rating': '6: marginally above the acceptanc...,Reject,
4,eUgS9Ig8JG,9491,SaNN: Simple Yet Powerful Simplicial-aware Neu...,Simplicial neural networks (SNNs) are deep mod...,"[Graph Neural Networks, Higher-order Represent...",/pdf/b5b2e785dec69b9ea0c8b01d6e2eca5896246cce.pdf,4,"[{'rating': '8: accept, good paper', 'confiden...",Accept (spotlight),


In [36]:
df.columns

Index(['paper_id', 'number', 'title', 'abstract', 'keywords', 'pdf',
       'n_reviews', 'reviews', 'decision', 'decision_comment'],
      dtype='object')

In [37]:
df["reviews"].head(1)

0    [{'rating': '3: reject, not good enough', 'con...
Name: reviews, dtype: object

In [38]:

# Load

# 1) One row per reviewer
tmp = df.explode("reviews", ignore_index=True)

# 2) Expand review dict into columns
rev = pd.json_normalize(tmp["reviews"])

# 3) Build clean table with requested columns
out = pd.DataFrame({
    "rating_raw": rev.get("rating"),
    "confidence_raw": rev.get("confidence"),
    "summary": rev.get("summary"),
    "strengths": rev.get("strengths"),
    "weaknesses": rev.get("weaknesses"),
})

# 4) Split rating into score + meaning
# Example: "6: marginally above the acceptance threshold"
rating_split = out["rating_raw"].astype(str).str.split(":", n=1, expand=True)
out["rating_score"] = pd.to_numeric(rating_split[0].str.strip(), errors="coerce")
out["rating_meaning"] = rating_split[1].str.strip() if rating_split.shape[1] > 1 else None

# 5) Split confidence into score + meaning
# Example: "4: You are confident in your assessment..."
conf_split = out["confidence_raw"].astype(str).str.split(":", n=1, expand=True)
out["confidence_score"] = pd.to_numeric(conf_split[0].str.strip(), errors="coerce")
out["confidence_meaning"] = conf_split[1].str.strip() if conf_split.shape[1] > 1 else None

# 6) Final column order
out = out[
    [
        "rating_score",
        "rating_meaning",
        "confidence_score",
        "confidence_meaning",
        "summary",
        "strengths",
        "weaknesses",
    ]
]

print(out.head())
# Optional save:
# out.to_csv("reviews_flattened.csv", index=False)

   rating_score                             rating_meaning  confidence_score  \
0           3.0                    reject, not good enough               4.0   
1           3.0                    reject, not good enough               3.0   
2           6.0  marginally above the acceptance threshold               3.0   
3           6.0  marginally above the acceptance threshold               3.0   
4           8.0                         accept, good paper               3.0   

                                  confidence_meaning  \
0  You are confident in your assessment, but not ...   
1  You are fairly confident in your assessment. I...   
2  You are fairly confident in your assessment. I...   
3  You are fairly confident in your assessment. I...   
4  You are fairly confident in your assessment. I...   

                                             summary  \
0  The paper aims at predicting steady-state comp...   
1  The paper tested the idea of using MPGNN or Gr...   
2  The paper l

In [39]:
out.head(10)

,rating_score,rating_meaning,confidence_score,confidence_meaning,summary,strengths,weaknesses
0,3.0,"reject, not good enough",4.0,"You are confident in your assessment, but not ...",The paper aims at predicting steady-state comp...,Understanding how distinct bacteria form commu...,The proposed approach for using GNNs for bacte...
1,3.0,"reject, not good enough",3.0,You are fairly confident in your assessment. I...,The paper tested the idea of using MPGNN or Gr...,The presented comparison results with MLP-base...,1. The methodological contribution is limited ...
2,6.0,marginally above the acceptance threshold,3.0,You are fairly confident in your assessment. I...,The paper looks at modeling bacterial communit...,I found the paper interesting and I think the ...,While I enjoyed reading something on the outsk...
3,6.0,marginally above the acceptance threshold,3.0,You are fairly confident in your assessment. I...,The study focuses on understanding the interac...,- Novel problem setup and the first use of GNN...,- Methodological novelty is limited since it i...
4,8.0,"accept, good paper",3.0,You are fairly confident in your assessment. I...,This paper considers the problem of making pre...,1. The results seem to be a significant advanc...,A comparison of the inference and query comple...
5,6.0,marginally above the acceptance threshold,2.0,"You are willing to defend your assessment, but...",This work proposes a retrieval-augmented deep ...,1. The extensive amount of open-sourcing and e...,1. Paper doesn't go into detail describing dif...
6,6.0,marginally above the acceptance threshold,3.0,You are fairly confident in your assessment. I...,"The paper introduces TabR, a retrieval-augment...",1. TabR demonstrates superior performance comp...,"1. Some aspects are not clear, see the questio..."
7,3.0,"reject, not good enough",4.0,"You are confident in your assessment, but not ...",The authors meticulously designed a supervised...,"- As emphasized by the authors, their method h...",- The motivations behind the module designs ar...
8,6.0,marginally above the acceptance threshold,2.0,"You are willing to defend your assessment, but...",The paper introduces a novel approach called N...,"The strengths of the paper ""Neural Evolutionar...","While the paper on ""Neural Evolutionary Kernel..."
9,5.0,marginally below the acceptance threshold,4.0,"You are confident in your assessment, but not ...",This paper aims to tackle solving partial diff...,- Nice abstract that motivates the need for PD...,- The authors should define earlier what they ...


In [56]:
# %pip install sentence-transformers
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer


# 2) Keep paper metadata
paper_cols = ["paper_id", "title", "abstract"]

# 3) Flatten reviews
tmp = df[paper_cols + ["reviews"]].explode("reviews", ignore_index=True)
rev = pd.json_normalize(tmp["reviews"])
long = pd.concat([tmp.drop(columns=["reviews"]), rev], axis=1)

# 4) Parse rating/confidence numeric scores from strings like "6: ..."
r = long["rating"].astype(str).str.split(":", n=1, expand=True)
long["rating_score"] = pd.to_numeric(r[0].str.strip(), errors="coerce")

c = long["confidence"].astype(str).str.split(":", n=1, expand=True)
long["confidence_score"] = pd.to_numeric(c[0].str.strip(), errors="coerce")

# 5) Clean text fields
for col in ["summary", "strengths", "weaknesses"]:
    if col not in long.columns:
        long[col] = ""
    long[col] = long[col].fillna("").astype(str).str.strip()

# 6) Numeric aggregates
agg = long.groupby("paper_id", as_index=False).agg(
    avg_rating=("rating_score", "mean"),
    std_rating=("rating_score", "std"),
    avg_confidence=("confidence_score", "mean"),
    std_confidence=("confidence_score", "std"),
    n_reviews=("rating_score", "count"),
)
agg["std_rating"] = agg["std_rating"].fillna(0.0)
agg["std_confidence"] = agg["std_confidence"].fillna(0.0)

# 7) Text aggregates
SEP = " [SEP] "

def join_nonempty(s):
    vals = [x for x in s if isinstance(x, str) and x.strip()]
    return SEP.join(vals) if vals else ""

text_agg = long.groupby("paper_id", as_index=False).agg(
    all_review_summaries=("summary", join_nonempty),
    all_strengths=("strengths", join_nonempty),
    all_weaknesses=("weaknesses", join_nonempty),
)

# 8) Merge core final table
paper_meta = df[paper_cols].drop_duplicates(subset=["paper_id"])
final_df = (
    paper_meta
    .merge(agg, on="paper_id", how="left")
    .merge(text_agg, on="paper_id", how="left")
)

# 9) Embeddings
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

summary_emb = model.encode(final_df["all_review_summaries"].fillna("").tolist(), normalize_embeddings=True)
strength_emb = model.encode(final_df["all_strengths"].fillna("").tolist(), normalize_embeddings=True)
weakness_emb = model.encode(final_df["all_weaknesses"].fillna("").tolist(), normalize_embeddings=True)

# store as JSON strings (CSV-safe)
final_df["summary_embedding"] = [json.dumps(v.tolist()) for v in summary_emb]
final_df["strength_embedding"] = [json.dumps(v.tolist()) for v in strength_emb]
final_df["weakness_embedding"] = [json.dumps(v.tolist()) for v in weakness_emb]

# 10) Exact final column order requested
final_df = final_df[
    [
        "paper_id",
        "title",
        "abstract",
        "avg_rating",
        "std_rating",
        "avg_confidence",
        "std_confidence",
        "n_reviews",
        "all_review_summaries",
        "all_strengths",
        "all_weaknesses",
        "summary_embedding",
        "strength_embedding",
        "weakness_embedding",
    ]
]

final_df.head()
# final_df.to_csv("final_dataset.csv", index=False)
# final_df.to_parquet("final_dataset.parquet", index=False)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10231.71it/s]


,paper_id,title,abstract,avg_rating,std_rating,avg_confidence,std_confidence,n_reviews,all_review_summaries,all_strengths,all_weaknesses,summary_embedding,strength_embedding,weakness_embedding
0,cXs5md5wAq,Modelling Microbial Communities with Graph Neu...,Understanding the interactions and interplay o...,4.50,1.732051,3.25,0.500000,4,The paper aims at predicting steady-state comp...,Understanding how distinct bacteria form commu...,The proposed approach for using GNNs for bacte...,"[-0.06159112602472305, -0.09591381251811981, -...","[-0.06317638605833054, -0.09650619328022003, 0...","[-0.10731928795576096, -0.07615976780653, -0.0..."
1,rhgIgTSSxW,TabR: Tabular Deep Learning Meets Nearest Neig...,Deep learning (DL) models for tabular data pro...,5.75,2.061553,3.00,0.816497,4,This paper considers the problem of making pre...,1. The results seem to be a significant advanc...,A comparison of the inference and query comple...,"[-0.10334067791700363, -0.12188716977834702, -...","[-0.09051284193992615, -0.07916262745857239, 0...","[-0.04136057198047638, -0.01884959265589714, 0..."
2,kKRbAY4CXv,Neural Evolutionary Kernel Method: A Knowledge...,Numerical solution of partial differential equ...,4.25,1.500000,3.25,0.957427,4,The paper introduces a novel approach called N...,"The strengths of the paper ""Neural Evolutionar...","While the paper on ""Neural Evolutionary Kernel...","[-0.09763363748788834, 0.009315716102719307, 0...","[-0.12222994863986969, 0.02320857159793377, 0....","[-0.14362375438213348, -0.00563540356233716, 0..."
3,ApjY32f3Xr,PINNacle: A Comprehensive Benchmark of Physics...,While significant progress has been made on Ph...,5.25,1.500000,3.50,1.000000,4,This paper provides both a collection of bench...,Providing any meaningful benchmark to the comm...,While providing a benchmark data set to the co...,"[-0.09181807935237885, -0.09487534314393997, 0...","[-0.08804965764284134, -0.10004816204309464, 0...","[-0.033457834273576736, -0.04419556260108948, ..."
4,eUgS9Ig8JG,SaNN: Simple Yet Powerful Simplicial-aware Neu...,Simplicial neural networks (SNNs) are deep mod...,7.00,1.154701,3.25,0.957427,4,"The paper describes an efficient, and effectiv...",1. I am impressed by the clarity of presentati...,I only have 1 important concern:\n\n1. Althoug...,"[-0.003033256623893976, -0.06415839493274689, ...","[-0.06320379674434662, -0.020994309335947037, ...","[-0.07742684334516525, 0.011792046017944813, -..."


In [62]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer

# 1) Keep paper metadata
paper_cols = ["paper_id", "title", "abstract"]

# 2) Flatten reviews
tmp = df[paper_cols + ["reviews"]].explode("reviews", ignore_index=True)
rev = pd.json_normalize(tmp["reviews"])
long = pd.concat([tmp.drop(columns=["reviews"]), rev], axis=1)

# 3) Parse rating/confidence score + label
r = long["rating"].astype(str).str.split(":", n=1, expand=True)
long["rating_score"] = pd.to_numeric(r[0].str.strip(), errors="coerce")
long["rating_label"] = r[1].fillna("").str.strip() if r.shape[1] > 1 else ""

c = long["confidence"].astype(str).str.split(":", n=1, expand=True)
long["confidence_score"] = pd.to_numeric(c[0].str.strip(), errors="coerce")
long["confidence_label"] = c[1].fillna("").str.strip() if c.shape[1] > 1 else ""

# 4) Clean text fields
for col in ["summary", "strengths", "weaknesses", "rating_label", "confidence_label"]:
    if col not in long.columns:
        long[col] = ""
    long[col] = long[col].fillna("").astype(str).str.strip()

# 5) Numeric aggregates
agg = long.groupby("paper_id", as_index=False).agg(
    avg_rating=("rating_score", "mean"),
    std_rating=("rating_score", "std"),
    avg_confidence=("confidence_score", "mean"),
    std_confidence=("confidence_score", "std"),
    n_reviews=("rating_score", "count"),
)
agg["std_rating"] = agg["std_rating"].fillna(0.0)
agg["std_confidence"] = agg["std_confidence"].fillna(0.0)

# 6) Text aggregates
SEP = " [SEP] "

def join_nonempty(s):
    vals = [x for x in s if isinstance(x, str) and x.strip()]
    return SEP.join(vals) if vals else ""

text_agg = long.groupby("paper_id", as_index=False).agg(
    all_review_summaries=("summary", join_nonempty),
    all_strengths=("strengths", join_nonempty),
    all_weaknesses=("weaknesses", join_nonempty),
    rating_labels_text=("rating_label", join_nonempty),
    confidence_labels_text=("confidence_label", join_nonempty),
)

# 7) Merge core table
paper_meta = df[paper_cols].drop_duplicates(subset=["paper_id"])
final_df = (
    paper_meta
    .merge(agg, on="paper_id", how="left")
    .merge(text_agg, on="paper_id", how="left")
)

# 8) Embeddings
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

summary_emb = model.encode(final_df["all_review_summaries"].fillna("").tolist(), normalize_embeddings=True)
strength_emb = model.encode(final_df["all_strengths"].fillna("").tolist(), normalize_embeddings=True)
weakness_emb = model.encode(final_df["all_weaknesses"].fillna("").tolist(), normalize_embeddings=True)

final_df["summary_embedding"] = [json.dumps(v.tolist()) for v in summary_emb]
final_df["strength_embedding"] = [json.dumps(v.tolist()) for v in strength_emb]
final_df["weakness_embedding"] = [json.dumps(v.tolist()) for v in weakness_emb]

# 9) Final columns
final_df = final_df[
    [
        "paper_id",
        "title",
        "abstract",
        "avg_rating",
        "std_rating",
        "avg_confidence",
        "std_confidence",
        "n_reviews",
        "rating_labels_text",
        "confidence_labels_text",
        "all_review_summaries",
        "all_strengths",
        "all_weaknesses",
        "summary_embedding",
        "strength_embedding",
        "weakness_embedding",
    ]
]

final_df.head()
# final_df.to_csv("final_dataset.csv", index=False)
# final_df.to_parquet("final_dataset.parquet", index=False)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25603.82it/s]


,paper_id,title,abstract,avg_rating,std_rating,avg_confidence,std_confidence,n_reviews,rating_labels_text,confidence_labels_text,all_review_summaries,all_strengths,all_weaknesses,summary_embedding,strength_embedding,weakness_embedding
0,cXs5md5wAq,Modelling Microbial Communities with Graph Neu...,Understanding the interactions and interplay o...,4.50,1.732051,3.25,0.500000,4,"reject, not good enough [SEP] reject, not good...","You are confident in your assessment, but not ...",The paper aims at predicting steady-state comp...,Understanding how distinct bacteria form commu...,The proposed approach for using GNNs for bacte...,"[-0.06159112602472305, -0.09591381251811981, -...","[-0.06317638605833054, -0.09650619328022003, 0...","[-0.10731928795576096, -0.07615976780653, -0.0..."
1,rhgIgTSSxW,TabR: Tabular Deep Learning Meets Nearest Neig...,Deep learning (DL) models for tabular data pro...,5.75,2.061553,3.00,0.816497,4,"accept, good paper [SEP] marginally above the ...",You are fairly confident in your assessment. I...,This paper considers the problem of making pre...,1. The results seem to be a significant advanc...,A comparison of the inference and query comple...,"[-0.10334067791700363, -0.12188716977834702, -...","[-0.09051284193992615, -0.07916262745857239, 0...","[-0.04136057198047638, -0.01884959265589714, 0..."
2,kKRbAY4CXv,Neural Evolutionary Kernel Method: A Knowledge...,Numerical solution of partial differential equ...,4.25,1.500000,3.25,0.957427,4,marginally above the acceptance threshold [SEP...,"You are willing to defend your assessment, but...",The paper introduces a novel approach called N...,"The strengths of the paper ""Neural Evolutionar...","While the paper on ""Neural Evolutionary Kernel...","[-0.09763363748788834, 0.009315716102719307, 0...","[-0.12222994863986969, 0.02320857159793377, 0....","[-0.14362375438213348, -0.00563540356233716, 0..."
3,ApjY32f3Xr,PINNacle: A Comprehensive Benchmark of Physics...,While significant progress has been made on Ph...,5.25,1.500000,3.50,1.000000,4,marginally above the acceptance threshold [SEP...,You are fairly confident in your assessment. I...,This paper provides both a collection of bench...,Providing any meaningful benchmark to the comm...,While providing a benchmark data set to the co...,"[-0.09181807935237885, -0.09487534314393997, 0...","[-0.08804965764284134, -0.10004816204309464, 0...","[-0.033457834273576736, -0.04419556260108948, ..."
4,eUgS9Ig8JG,SaNN: Simple Yet Powerful Simplicial-aware Neu...,Simplicial neural networks (SNNs) are deep mod...,7.00,1.154701,3.25,0.957427,4,"accept, good paper [SEP] accept, good paper [S...","You are confident in your assessment, but not ...","The paper describes an efficient, and effectiv...",1. I am impressed by the clarity of presentati...,I only have 1 important concern:\n\n1. Althoug...,"[-0.003033256623893976, -0.06415839493274689, ...","[-0.06320379674434662, -0.020994309335947037, ...","[-0.07742684334516525, 0.011792046017944813, -..."


In [61]:
print(final_df.loc[0, "all_review_summaries"])




The paper aims at predicting steady-state composition of microbial communities from the gene content of their genomes using graph neural networks. [SEP] The paper tested the idea of using MPGNN or GraphSAGE to learn generalizable microbial community steady-state dynamics. The proposed models were tested on  simulated and previous publicly available microbial datasets and compared with the MLP-based implementation to show the effectiveness, with the discussions on the generalizability of GNN-based implementations. [SEP] The paper looks at modeling bacterial communities and their interactions using graph neural networks (GNNs). They rely on two open datasets, total n = 552 samples. The authors have downloaded genomes for the bacteria that was converted to growth encodings. To address the issue with limited data the authors also used a simulator based on the Lotka-Volterra model. They compare three different models, MLP as the standard, GNNs and MPGNN. Using GNN/MPGNN the authors were abl

In [42]:
import json


paper_cols = ["paper_id", "number", "title", "abstract", "keywords", "pdf", "n_reviews"]

df["keywords"] = df["keywords"].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x
)

tmp = df[paper_cols + ["reviews"]].explode("reviews", ignore_index=True)
rev = pd.json_normalize(tmp["reviews"])
long = pd.concat([tmp.drop(columns=["reviews"]), rev], axis=1)

r = long["rating"].astype(str).str.split(":", n=1, expand=True)
long["rating_score"] = pd.to_numeric(r[0].str.strip(), errors="coerce")
long["rating_meaning"] = r[1].str.strip()

c = long["confidence"].astype(str).str.split(":", n=1, expand=True)
long["confidence_score"] = pd.to_numeric(c[0].str.strip(), errors="coerce")
long["confidence_meaning"] = c[1].str.strip()

long["review_idx"] = long.groupby("paper_id").cumcount() + 1

# keep only first 6 reviews per paper
long = long[long["review_idx"] <= 6]

review_cols = [
    "rating_score", "rating_meaning",
    "confidence_score", "confidence_meaning",
    "summary", "strengths", "weaknesses"
]

wide = long.pivot(index="paper_id", columns="review_idx", values=review_cols)

# force exactly 6 slots even if some papers have fewer
target_cols = pd.MultiIndex.from_product([review_cols, range(1, 7)])
wide = wide.reindex(columns=target_cols)

wide.columns = [f"{col}_{idx}" for col, idx in wide.columns]
wide = wide.reset_index()

paper_meta = df[paper_cols].drop_duplicates(subset=["paper_id"])
final_df = paper_meta.merge(wide, on="paper_id", how="left")

final_df.head()
# final_df.to_csv("papers_reviews_wide_6.csv", index=False)

,paper_id,number,title,abstract,keywords,pdf,n_reviews,rating_score_1,rating_score_2,rating_score_3,...,strengths_3,strengths_4,strengths_5,strengths_6,weaknesses_1,weaknesses_2,weaknesses_3,weaknesses_4,weaknesses_5,weaknesses_6
0,cXs5md5wAq,9504,Modelling Microbial Communities with Graph Neu...,Understanding the interactions and interplay o...,"[""graph neural networks"", ""microbial communiti...",/pdf/a4578db3b369ee02db6e42b64d333e578e1b692e.pdf,4,3.0,3.0,6.0,...,I found the paper interesting and I think the ...,- Novel problem setup and the first use of GNN...,NaN,NaN,The proposed approach for using GNNs for bacte...,1. The methodological contribution is limited ...,While I enjoyed reading something on the outsk...,- Methodological novelty is limited since it i...,NaN,NaN
1,rhgIgTSSxW,9502,TabR: Tabular Deep Learning Meets Nearest Neig...,Deep learning (DL) models for tabular data pro...,"[""tabular"", ""tabular data"", ""architecture"", ""d...",/pdf/178e173a880d7872c0a79d88e005426c20501329.pdf,4,8.0,6.0,6.0,...,1. TabR demonstrates superior performance comp...,"- As emphasized by the authors, their method h...",NaN,NaN,A comparison of the inference and query comple...,1. Paper doesn't go into detail describing dif...,"1. Some aspects are not clear, see the questio...",- The motivations behind the module designs ar...,NaN,NaN
2,kKRbAY4CXv,9498,Neural Evolutionary Kernel Method: A Knowledge...,Numerical solution of partial differential equ...,"[""Numerical PDE"", ""structure preserving neural...",/pdf/c330ae354c1b65e4afaa1f53a1ca188d24fcf27f.pdf,4,6.0,5.0,3.0,...,NEKM can be combined with time discretization ...,- The paper is well-written and easy-to-follow...,NaN,NaN,"While the paper on ""Neural Evolutionary Kernel...",- The authors should define earlier what they ...,While NEKM is claimed to work in complex domai...,- It seems the method heavily relies on the cl...,NaN,NaN
3,ApjY32f3Xr,9493,PINNacle: A Comprehensive Benchmark of Physics...,While significant progress has been made on Ph...,"[""PINN"", ""machine learning"", ""physics-informed...",/pdf/49840b2f19f2bbeff0c1539d86c876d01140da89.pdf,4,6.0,6.0,3.0,...,1. The paper is overall well written and easy ...,"This paper stands out with several merits, acc...",NaN,NaN,While providing a benchmark data set to the co...,1. I'm saying out of respect to the author's w...,1. The paper lacks technical novelty to be con...,The paper has some areas it could improve on.\...,NaN,NaN
4,eUgS9Ig8JG,9491,SaNN: Simple Yet Powerful Simplicial-aware Neu...,Simplicial neural networks (SNNs) are deep mod...,"[""Graph Neural Networks"", ""Higher-order Repres...",/pdf/b5b2e785dec69b9ea0c8b01d6e2eca5896246cce.pdf,4,8.0,8.0,6.0,...,"- The goal of the work, achieving better scala...","1. For the most part, this paper is well-writt...",NaN,NaN,I only have 1 important concern:\n\n1. Althoug...,1. It is unclear for a research with limited e...,- Runtime and asymptotic comparisons in this w...,1. Certain definitions regarding the types of ...,NaN,NaN


In [ ]:
final_df["confidence_meaning_2"].head(10)

0    You are fairly confident in your assessment. I...
1    You are willing to defend your assessment, but...
2    You are confident in your assessment, but not ...
3    You are fairly confident in your assessment. I...
4    You are willing to defend your assessment, but...
5                                                  NaN
6    You are absolutely certain about your assessme...
7    You are fairly confident in your assessment. I...
8    You are confident in your assessment, but not ...
9    You are fairly confident in your assessment. I...
Name: confidence_meaning_2, dtype: object

In [51]:
import json


paper_cols = ["paper_id", "number", "title", "abstract", "keywords", "pdf", "n_reviews"]

df["keywords"] = df["keywords"].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x
)

tmp = df[paper_cols + ["reviews"]].explode("reviews", ignore_index=True)
rev = pd.json_normalize(tmp["reviews"])
long = pd.concat([tmp.drop(columns=["reviews"]), rev], axis=1)

# Parse rating + confidence scores
r = long["rating"].astype(str).str.split(":", n=1, expand=True)
long["rating_score"] = pd.to_numeric(r[0].str.strip(), errors="coerce")

c = long["confidence"].astype(str).str.split(":", n=1, expand=True)
long["confidence_score"] = pd.to_numeric(c[0].str.strip(), errors="coerce")

# reviewer index
long["review_idx"] = long.groupby("paper_id").cumcount() + 1
long6 = long[long["review_idx"] <= 6].copy()

# Averages (single columns)
agg = long.groupby("paper_id", as_index=False).agg(
    avg_rating=("rating_score", "mean"),
    avg_confidence=("confidence_score", "mean")
)

# Keep summary/strengths/weaknesses per reviewer (1..6)
txt = long6.pivot(
    index="paper_id",
    columns="review_idx",
    values=["summary", "strengths", "weaknesses"]
)

target_cols = pd.MultiIndex.from_product(
    [["summary", "strengths", "weaknesses"], range(1, 7)]
)
txt = txt.reindex(columns=target_cols)

txt.columns = [f"{col}_{idx}" for col, idx in txt.columns]
txt = txt.reset_index()

paper_meta = df[paper_cols].drop_duplicates(subset=["paper_id"])

final_df = (
    paper_meta
    .merge(agg, on="paper_id", how="left")
    .merge(txt, on="paper_id", how="left")
)

final_df.columns
# final_df.to_csv("papers_avg_rating_confidence_with_summaries.csv", index=False)

Index(['paper_id', 'number', 'title', 'abstract', 'keywords', 'pdf',
       'n_reviews', 'avg_rating', 'avg_confidence', 'summary_1', 'summary_2',
       'summary_3', 'summary_4', 'summary_5', 'summary_6', 'strengths_1',
       'strengths_2', 'strengths_3', 'strengths_4', 'strengths_5',
       'strengths_6', 'weaknesses_1', 'weaknesses_2', 'weaknesses_3',
       'weaknesses_4', 'weaknesses_5', 'weaknesses_6'],
      dtype='object')